# 1. EDA temporal y auditoría de fuga — IEEE-CIS

## Resumen y propósito
Identificamos el desbalance, patrones temporales, distribuciones relevantes, faltantes y posibles fugas de información antes de modelar. **Fuente:** `train_transaction` y `train_identity` para resultados con etiqueta; `test_*` solo para distribuciones sin etiqueta. La primera tabla indica el tamaño y alcance de la corrida actual.

## Método y supuestos
`TransactionDT` es tiempo relativo. Las cifras de fraude provienen solo del archivo train. El muestreo de montos usa semilla fija y se usa únicamente para el gráfico. Ninguna diferencia descriptiva se interpreta como causalidad.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

FINAL_ROOT = Path.cwd() if (Path.cwd() / 'v01_features.json').exists() else Path.cwd() / 'final'
__file__ = str(FINAL_ROOT / 'common.py')
V01_FEATURES = ['V258', 'C5', 'D1n', 'D3', 'C13', 'C1', 'C14', 'card1', 'D2n', 'card2', 'TransactionAmt', 'D15n', 'addr1', 'D2', 'D10n', 'C11', 'card6_label', 'P_emaildomain_label', 'C6', 'card5', 'D4', 'M4_freq', 'V257', 'D15', 'C2', 'C8', 'dist1', 'card3', 'card6_freq', 'C9', 'P_emaildomain_freq', 'D8', 'M4_label', 'M5_label', 'D1', 'D10', 'C10', 'M5_freq', 'id_02', 'C4', 'DT_hour', 'ProductCD_label', 'R_emaildomain_label', 'V189', 'V45', 'V243', 'D5', 'M6_freq', 'DeviceInfo_label', 'id_20', 'TransactionAmt_log1p', 'D11', 'id_01', 'C12', 'V87', 'DeviceInfo_freq', 'V78', 'V201', 'id_19', 'id_13', 'V140', 'ProductCD_freq', 'missing_D_count', 'V262', 'id_05', 'V38', 'M6_label', 'card4_label', 'dist2', 'V156', 'id_14', 'id_06', 'M3_label', 'missing_V_count', 'R_emaildomain_freq', 'V149', 'id_17', 'card4_freq', 'V94', 'D13', 'V37', 'id_18', 'DeviceType_freq', 'V74', 'D6', 'id_32', 'V58', 'D14', 'id_09', 'D9', 'D12', 'V86', 'addr2', 'C7', 'C3', 'V64', 'V170', 'id_03', 'D7', 'V44', 'V47', 'V217', 'missing_core_count', 'V52', 'V23', 'TransactionAmt_outlier_iqr', 'V79', 'M3_freq', 'V148', 'V157', 'M2_label', 'V33', 'V171', 'M9_label', 'M7_label', 'missing_id_count', 'V57', 'V219', 'V81', 'V39', 'M8_label', 'V154', 'V199', 'M9_freq', 'V232', 'V63', 'V34', 'V177', 'id_11', 'V228', 'V233', 'V230', 'V40', 'V158', 'V146', 'V231', 'id_04', 'V60', 'V155', 'V200', 'V51', 'V188', 'id_24', 'V190', 'DeviceType_label', 'V50', 'M8_freq', 'V85', 'V252', 'V73', 'M2_freq', 'id_21', 'V147', 'V197', 'V153', 'M7_freq', 'V16', 'V72', 'V84', 'V246', 'V43', 'V15', 'id_08', 'id_26', 'id_25', 'V244', 'id_07', 'V247', 'M1_freq', 'V93', 'V42', 'V71', 'V18', 'V242', 'V92', 'V32', 'id_10', 'V80', 'M1_label', 'V21', 'V17', 'V22', 'V31', 'has_identity', 'id_22']
RUN_FULL = (os.name == 'posix' and Path('/kaggle/working').exists()) or os.getenv('PTDIA_RUN_FULL') == '1'
print('Ejecución completa:', RUN_FULL)

### Implementación reproducible
Las funciones siguientes son código ejecutable del proyecto. Se agrupan por responsabilidad y se pueden ejecutar de arriba abajo sin archivos auxiliares.

In [ ]:
"""Shared, deterministic IEEE-CIS helpers for the four final notebooks.

The notebook entry points call these functions. Nothing is fitted on future
rows: a preprocessing instance is created for each historical training window.
"""

from __future__ import annotations

import gc
import gzip
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.metrics import average_precision_score, precision_score, recall_score

DAY = 86400
WEEK = 7 * DAY
SEED = 42
CAT_COLS = ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo"] + [f"M{i}" for i in range(1, 10)]
MONITOR_COLS = ["TransactionAmt", "D1n", "D2n", "D10n", "D15n", "C9", "id_02", "id_20", "D11", "id_01"]


def paths(stage: str):
    here = Path(__file__).resolve().parent
    candidates = [
        Path("/kaggle/input/ieee-fraud-detection"),
        Path("/kaggle/input/competitions/ieee-fraud-detection"),
        here.parent / "ieee-fraud-detection",
    ]
    data = next((p for p in candidates if (p / "train_transaction.csv").exists()), None)
    if data is None:
        raise FileNotFoundError("IEEE-CIS competition input unavailable")
    output = Path("/kaggle/working") if os.name == "posix" and Path("/kaggle/working").exists() else here / "kaggle" / stage / "outputs"
    output.mkdir(parents=True, exist_ok=True)
    (output / "plots").mkdir(exist_ok=True)
    return data, output


def manifest(output: Path, **values):
    target = output / "run_summary.json"
    current = json.loads(target.read_text()) if target.exists() else {}
    current.update(values)
    target.write_text(json.dumps(current, indent=2, default=float), encoding="utf-8")


def save_plot(output: Path, name: str):
    plt.tight_layout()
    plt.savefig(output / "plots" / name, dpi=140, bbox_inches="tight")
    plt.close()


def load_data(data: Path, include_test: bool = False):
    def one(prefix: str):
        txn = pd.read_csv(data / f"{prefix}_transaction.csv", low_memory=False)
        identity = pd.read_csv(data / f"{prefix}_identity.csv", low_memory=False)
        # IEEE-CIS test identity uses id-01 style names, while train uses id_01.
        identity.rename(columns=lambda c: c.replace("id-", "id_") if c.startswith("id-") else c, inplace=True)
        if txn.TransactionID.duplicated().any() or identity.TransactionID.duplicated().any():
            raise ValueError("TransactionID must be unique in each source")
        out = txn.merge(identity, on="TransactionID", how="left", validate="one_to_one", indicator="_identity_join")
        del txn, identity
        gc.collect()
        out["has_identity_join"] = out["_identity_join"].eq("both").astype("int8")
        out.drop(columns="_identity_join", inplace=True)
        out.sort_values(["TransactionDT", "TransactionID"], inplace=True)
        out.reset_index(drop=True, inplace=True)
        return out

    train = one("train")
    test = one("test") if include_test else None
    return train, test


def split_bounds(df: pd.DataFrame):
    return float(df.TransactionDT.quantile(0.70)), float(df.TransactionDT.quantile(0.85))


def split_name(dt: pd.Series, bounds: tuple[float, float]):
    return np.where(dt <= bounds[0], "train", np.where(dt <= bounds[1], "valid", "holdout"))


def row_features(df: pd.DataFrame):
    """Only own-row information. Fitted statistics are added by WindowEncoder."""
    out = df.copy()
    day = (out.TransactionDT // DAY).astype("int16")
    out["DT_hour"] = ((out.TransactionDT // 3600) % 24).astype("int8")
    out["DT_day_index"] = day
    out["DT_week_index"] = (day // 7).astype("int16")
    out["has_identity"] = out["has_identity_join"].astype("int8")
    out["TransactionAmt_log1p"] = np.log1p(out.TransactionAmt.clip(lower=0)).astype("float32")
    for family in ("V", "D", "id_"):
        cols = [c for c in out if c.startswith(family) and (family == "id_" or c[len(family):].isdigit())]
        out[f"missing_{family.replace('_', '')}_count"] = out[cols].isna().sum(axis=1).astype("int16") if cols else 0
    core = [c for c in ["card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"] if c in out]
    out["missing_core_count"] = out[core].isna().sum(axis=1).astype("int8")
    for col in ["D1", "D2", "D10", "D15"]:
        out[f"{col}n"] = (out[col] - day).astype("float32")
    return out


def candidate_uid(df: pd.DataFrame):
    parts = df[["card1", "card2", "card3", "card5", "addr1", "addr2"]].astype("string").fillna("NA")
    parts = parts.copy()
    parts["D1_anchor"] = (df.D1 - df.TransactionDT // DAY).round(0).astype("string").fillna("NA")
    return pd.util.hash_pandas_object(parts, index=False).astype("uint64")

In [ ]:
def feature_list():
    here = Path(__file__).resolve().parent
    for path in (here / "v01_features.json", here / "input" / "v01_features.json"):
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    if "V01_FEATURES" in globals():
        return list(globals()["V01_FEATURES"])
    raise FileNotFoundError("v01_features.json is required")


@dataclass
class WindowEncoder:
    features: list[str]
    q_low: float = 0.0
    q_high: float = 0.0
    maps: dict | None = None
    freqs: dict | None = None

    def fit(self, train: pd.DataFrame):
        q1, q3 = train.TransactionAmt.quantile([.25, .75])
        self.q_low, self.q_high = float(q1 - 1.5 * (q3 - q1)), float(q3 + 1.5 * (q3 - q1))
        self.maps, self.freqs = {}, {}
        for col in CAT_COLS:
            if col not in train:
                continue
            s = train[col].astype("string").fillna("__MISSING__")
            self.maps[col] = {v: i for i, v in enumerate(s.unique())}
            self.freqs[col] = (s.value_counts(dropna=False) / len(s)).to_dict()
        return self

    def transform(self, rows: pd.DataFrame):
        cols = {}
        for name in self.features:
            if name == "TransactionAmt_outlier_iqr":
                cols[name] = ((rows.TransactionAmt < self.q_low) | (rows.TransactionAmt > self.q_high)).astype("float32")
            elif name.endswith("_label") and name[:-6] in self.maps:
                col = name[:-6]
                cols[name] = rows[col].astype("string").fillna("__MISSING__").map(self.maps[col]).fillna(-1).astype("float32")
            elif name.endswith("_freq") and name[:-5] in self.freqs:
                col = name[:-5]
                cols[name] = rows[col].astype("string").fillna("__MISSING__").map(self.freqs[col]).fillna(0).astype("float32")
            elif name in rows:
                cols[name] = pd.to_numeric(rows[name], errors="coerce").astype("float32")
            else:
                cols[name] = pd.Series(np.nan, index=rows.index, dtype="float32")
        return pd.DataFrame(cols, index=rows.index).replace([np.inf, -np.inf], np.nan)


def psi(reference, current, bins=10):
    a = np.asarray(reference, dtype=float)
    b = np.asarray(current, dtype=float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 100 or len(b) < 100:
        return np.nan
    edges = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0 if np.isclose(np.mean(a), np.mean(b)) else np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    pa = np.clip(np.histogram(a, edges)[0] / len(a), 1e-6, 1)
    pb = np.clip(np.histogram(b, edges)[0] / len(b), 1e-6, 1)
    return float(np.sum((pb - pa) * np.log(pb / pa)))


def safe_ap(y, pred):
    return float(average_precision_score(y, pred)) if len(y) and len(np.unique(y)) == 2 else np.nan


def evaluation_blocks(df: pd.DataFrame, bounds: tuple[float, float]):
    """Seven-day blocks clipped to each split; first valid week is label warm-up."""
    blocks = []
    for split, start, end in (("valid", bounds[0], bounds[1]), ("holdout", bounds[1], float(df.TransactionDT.max()) + 1)):
        index = 0
        t = start
        while t < end:
            u = min(t + WEEK, end)
            # First validation week warms up the seven-day label delay.
            if not (split == "valid" and index == 0):
                mask = (df.TransactionDT > t) & (df.TransactionDT <= u) if split == "valid" else (df.TransactionDT > t) & (df.TransactionDT < u)
                ix = df.index[mask].to_numpy()
                if len(ix):
                    blocks.append({"split": split, "block": index, "start": t, "end": u, "partial": u - t < WEEK, "indices": ix})
            t = u
            index += 1
    return blocks


def prediction_rows(df: pd.DataFrame, ix, uid_known, pred, model, strategy, window_days, block):
    return pd.DataFrame({
        "TransactionID": df.loc[ix, "TransactionID"].to_numpy(),
        "TransactionDT": df.loc[ix, "TransactionDT"].to_numpy(),
        "TransactionAmt": df.loc[ix, "TransactionAmt"].to_numpy(),
        "isFraud": df.loc[ix, "isFraud"].to_numpy(),
        "uid_known": uid_known,
        "score": np.asarray(pred),
        "model": model,
        "strategy": strategy,
        "window_days": window_days,
        "split": block["split"],
        "block": block["block"],
    })


def append_predictions(output: Path, rows: pd.DataFrame):
    path = output / "predictions.csv.gz"
    rows.to_csv(path, mode="at" if path.exists() else "wt", index=False, header=not path.exists(), compression="gzip")


def quick_cost(y, amount, score, review_fraction=.05, review_effectiveness=.8):
    """Offline same-capacity ranking proxy; operational queue is in policy notebook."""
    n = len(score)
    k = max(1, int(n * review_fraction))
    review = np.zeros(n, dtype=bool)
    review[np.argsort(score)[-k:]] = True
    loss = 4.41 * np.asarray(amount) * np.asarray(y)
    total = float(np.sum(np.where(review, 2 + (1 - review_effectiveness) * loss, loss)))
    return total / max(n, 1), float(recall_score(y, review, zero_division=0)), float(precision_score(y, review, zero_division=0))


def block_metrics(rows: pd.DataFrame, seconds=0.0, updated=False, device="none", psi_score=np.nan):
    y = rows.isFraud.to_numpy(dtype=int)
    p = rows.score.to_numpy(dtype=float)
    cost, recall, precision = quick_cost(y, rows.TransactionAmt, p)
    return {
        "model": rows.model.iloc[0], "strategy": rows.strategy.iloc[0], "window_days": rows.window_days.iloc[0],
        "split": rows.split.iloc[0], "block": int(rows.block.iloc[0]), "rows": len(rows),
        "fraud_rate": float(np.mean(y)), "pr_auc": safe_ap(y, p), "recall_at_5pct": recall,
        "precision_at_5pct": precision, "cost_per_txn_proxy": cost,
        "uid_known_pr_auc": safe_ap(y[rows.uid_known.to_numpy(dtype=bool)], p[rows.uid_known.to_numpy(dtype=bool)]),
        "uid_unknown_pr_auc": safe_ap(y[~rows.uid_known.to_numpy(dtype=bool)], p[~rows.uid_known.to_numpy(dtype=bool)]),
        "train_seconds": seconds, "updated": int(updated), "device": device, "psi_score": psi_score,
    }

In [ ]:
def run_eda():
    data, out = paths("eda")
    manifest(out, status="running", stage="load", source=str(data))
    train, test = load_data(data, include_test=True)
    bounds = split_bounds(train)
    train["week"] = (train.TransactionDT // WEEK).astype(int)
    prevalence = train.groupby("week", observed=True).isFraud.agg(["size", "sum", "mean"]).reset_index()
    prevalence.to_csv(out / "fraud_by_week.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 4))
    counts = train.isFraud.value_counts().reindex([0, 1])
    ax.bar(["Legitima", "Fraude"], counts.values, color=["#4c78a8", "#e45756"])
    ax.set(ylabel="Transacciones", title="Desbalance de clases en train etiquetado")
    for i, value in enumerate(counts.values):
        ax.text(i, value, f"{value:,}", ha="center", va="bottom")
    save_plot(out, "01_clases.png")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(prevalence.week, 100 * prevalence["mean"], marker="o", color="#e45756")
    ax.axvline(bounds[0] / WEEK, color="#f2a541", linestyle="--", label="70 %")
    ax.axvline(bounds[1] / WEEK, color="#8338ec", linestyle="--", label="85 %")
    ax.set(xlabel="Semana relativa", ylabel="Fraude (%)", title="Prevalencia semanal y cortes cronologicos")
    ax.legend()
    save_plot(out, "02_prevalencia_semanal.png")

    miss = pd.DataFrame({"train_missing": train.isna().mean(), "test_missing": test.isna().mean()})
    miss["delta"] = miss.test_missing - miss.train_missing
    miss.sort_values("train_missing", ascending=False).to_csv(out / "missingness.csv")
    chosen = miss.sort_values("train_missing", ascending=False).head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 6))
    y = np.arange(len(chosen))
    ax.barh(y - .2, chosen.train_missing * 100, height=.4, label="train")
    ax.barh(y + .2, chosen.test_missing * 100, height=.4, label="test")
    ax.set_yticks(y, chosen.index)
    ax.set(xlabel="Datos faltantes (%)", title="20 columnas con mas faltantes en train")
    ax.legend()
    save_plot(out, "03_faltantes.png")

    rng = np.random.default_rng(SEED)
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, frame, color in (("train", train, "#4c78a8"), ("test", test, "#e45756")):
        values = frame.TransactionAmt.dropna().to_numpy()
        values = rng.choice(values, min(80000, len(values)), replace=False)
        ax.hist(np.log1p(values.clip(min=0)), bins=65, density=True, alpha=.5, label=label, color=color)
    ax.set(xlabel="log(1 + monto)", ylabel="Densidad", title="Distribucion de montos; muestra aleatoria fija")
    ax.legend()
    save_plot(out, "04_montos.png")

    # Three numeric features ranked highly in the existing V01 importance.
    # Plot with fixed sampling and shared within-feature limits for readability.
    feature_stats = []
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
    for ax, col in zip(axes, ("V258", "C5", "D1")):
        bounds_col = train[col].dropna().quantile([.01, .99]).to_numpy()
        for value, label, color in ((0, "Legitima", "#4c78a8"), (1, "Fraude", "#e45756")):
            part = train.loc[train.isFraud == value, col]
            feature_stats.append({"feature": col, "class": label, "rows": len(part),
                                  "missing_rate": float(part.isna().mean()), "median": float(part.median())})
            values = part.dropna().to_numpy()
            if len(values) > 40000:
                values = rng.choice(values, 40000, replace=False)
            ax.hist(np.clip(values, *bounds_col), bins=45, density=True, alpha=.45,
                    label=label, color=color)
        ax.set(xlabel=col, ylabel="Densidad", title=f"{col}: clases en train")
    axes[0].legend()
    save_plot(out, "06_variables_v01.png")
    pd.DataFrame(feature_stats).to_csv(out / "important_feature_stats.csv", index=False)

    for col in ("ProductCD", "DeviceType"):
        group = train.groupby(col, observed=True, dropna=False).isFraud.agg(["size", "mean"]).sort_values("size", ascending=False).head(12)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(group.index.astype(str), group["mean"] * 100, color="#72b7b2")
        ax.axhline(train.isFraud.mean() * 100, color="black", linestyle="--", label="Tasa global")
        ax.set(xlabel=col, ylabel="Fraude (%)", title=f"Tasa descriptiva por {col}; train etiquetado")
        ax.legend()
        save_plot(out, f"05_riesgo_{col}.png")
        group.to_csv(out / f"risk_{col}.csv")

    # Row availability and preprocessing audits: do not assert that opaque V/C/D
    # columns can be proven leakage-free from public metadata.
    audit = [
        ("TransactionID", "excluir", "Llave unica y proxy de orden; no predictor."),
        ("TransactionDT / DT_day_index / DT_week_index", "excluir como predictor", "Solo orden, cortes y monitoreo."),
        ("TransactionAmt_outlier_iqr", "corregido", "Cuartiles ajustados en cada ventana; V01 los ajustaba sobre todas las filas."),
        ("Categorias y frecuencias", "controlado", "Mapas construidos solo con filas del entrenamiento permitido."),
        ("UID e historicos", "controlado con limite", "UID por atributos propios; agregados etiquetados solo del pasado, sin fila futura."),
        ("V/C/D anonimizadas", "no demostrable completamente", "Auditar disponibilidad en inferencia; el origen de cada campo no es publico."),
        ("Validacion / early stopping", "separacion", "La validacion selecciona decisiones; holdout final reservado para evaluacion."),
    ]
    pd.DataFrame(audit, columns=["feature_or_step", "status", "reason"]).to_csv(out / "leakage_audit.csv", index=False)
    summary = {
        "train_rows": len(train), "test_rows": len(test), "train_fraud_rate": float(train.isFraud.mean()),
        "train_day_min": float(train.TransactionDT.min() / DAY), "train_day_max": float(train.TransactionDT.max() / DAY),
        "test_day_min": float(test.TransactionDT.min() / DAY), "test_day_max": float(test.TransactionDT.max() / DAY),
        "test_has_target": "isFraud" in test, "train_cutoff": bounds[0], "valid_cutoff": bounds[1],
        "weekly_fraud_rate_min": float(prevalence["mean"].min()), "weekly_fraud_rate_max": float(prevalence["mean"].max()),
    }
    (out / "eda_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    manifest(out, status="complete", stage="done", **summary)
    return out, summary

### Ejecutar o cargar la exploración
En Kaggle se procesan los CSV completos; localmente se muestran las salidas descargadas.

In [ ]:
if RUN_FULL:
    output_dir, summary = run_eda()
else:
    output_dir = FINAL_ROOT / 'kaggle' / 'eda' / 'outputs'
    summary_file = output_dir / 'run_summary.json'
    if not summary_file.exists():
        raise FileNotFoundError(f'Faltan outputs descargados: {summary_file}')
    summary = json.loads(summary_file.read_text(encoding='utf-8'))
print('Salidas:', output_dir)
display(pd.Series(summary, name='valor').to_frame())

### Desbalance y evolución
La corrida completa registra 20 663 fraudes en 590 540 transacciones (3.499 %). La tasa semanal observada se mueve entre 1.85 % y 5.06 %: una métrica global oculta esos cambios. Las líneas verticales representan el split interno 70/15/15.

In [ ]:
for name in ['01_clases.png', '02_prevalencia_semanal.png']:
    path = output_dir / 'plots' / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Figura pendiente:', path)

### Faltantes, montos y segmentos
La identidad incompleta es parte del proceso de captura; train y test armonizan `id_`/`id-` antes de comparar. V258, C5 y D1 se eligieron entre variables bien posicionadas en la importancia de V01 y sus distribuciones se visualizan por clase sin usarlas como reglas de fraude. Los segmentos muestran asociaciones descriptivas y distintos denominadores.

In [ ]:
for name in ['03_faltantes.png', '04_montos.png', '06_variables_v01.png', '05_riesgo_ProductCD.png', '05_riesgo_DeviceType.png']:
    path = output_dir / 'plots' / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Figura pendiente:', path)

### Auditoría de leakage
Separamos campos excluidos, transformaciones corregidas y riesgo residual. Las columnas anonimizadas requieren confirmación operacional de disponibilidad al momento de inferencia.

In [ ]:
display(pd.read_csv(output_dir / 'leakage_audit.csv'))

## Conclusión y límite
El objetivo del EDA es justificar el split cronológico, la limpieza y el monitoreo. **No** demuestra que todas las columnas anonimizadas estén libres de fuga: esa garantía exigiría documentación de generación y disponibilidad en producción. La tabla de auditoría deja los controles verificables.